# Notebook 05: Gold Budget and Plan fact

## Runtime contract

- **Inputs/runtime:** current Silver budget and synthetic FAGLFLEXP projection; T001-derived company SCD2; Delta 3.2+ Spark for atomic additive migration.
- **Parameters:** `run_id` selects durable affected keys; blank performs full/manual upserts. `_SAP_EXTRACTION_CONFIG.tables[BUDGET].financeContext` must explicitly declare string `client` and `ledger`; no customer defaults are inferred.
- **Output:** all legacy columns/calculations/dimension relationships remain. Additive `RCLNT`, `RLDNR`, `RHCUR` extend the deterministic budget grain/key and permit currency-safe serving.
- **Currency identity:** FAGLFLEXP carries its declared RCLNT/RLDNR/RHCUR directly. BUDGET_FILE uses explicit metadata client/ledger and exactly one as-of T001-derived company WAERS; missing or ambiguous mappings fail.
- **Incrementality:** complete typed source keys scope all affected periods 1-16; master-only changes re-resolve matching period-date versions. Currency changes atomically re-key the existing source identity instead of leaving duplicate old-currency rows. Only journal-confirmed source removals are deleted.
- **Migration:** a legacy row must map to exactly one current extended identity. One schema-evolving MERGE changes legacy SKs and adds identity columns atomically. Zero/multiple candidates, inconsistent known identities, or overlapping legacy/new rows fail with backup/recovery guidance; nothing is guessed or silently duplicated.
- **Historical compatibility:** posting/period-date SCD2 semantics, precise decimals, carry-forward 000, special periods and legacy amount names are preserved. Synthetic FAGLFLEXP is not a native DDIC mapping.
- **Evidence:** per-grain monetary audit evidence never combines unlike currencies. Calendar fiscal mapping remains the existing demo assumption.


In [ ]:
run_id = ""  # Blank: full/manual upsert; pipeline supplies the durable run ID.


In [ ]:
from pyspark.sql import functions as F, types as T, Window
from delta.tables import DeltaTable
from functools import reduce
import datetime
import uuid

if not isinstance(run_id, str):
    raise TypeError("run_id must be a string")
run_id = run_id.strip()
BRZ_SLV_LH = "lh_brz_slv_finance"
GLD_LH = "lh_gld_finance"
SLV = f"{BRZ_SLV_LH}.slv"
AUD = f"{BRZ_SLV_LH}.audit"
DIM = f"{GLD_LH}.dim"
FCT = f"{GLD_LH}.fct"
HIGH_DATE = "9999-12-31"
BATCH_ID = run_id or str(uuid.uuid4())
RUN_TIMESTAMP = datetime.datetime.now(datetime.timezone.utc).replace(tzinfo=None)


In [ ]:
MASTER_SOURCE_KEYS = {
    "t001": ["MANDT", "BUKRS"],
    "ska1": ["MANDT", "KTOPL", "SAKNR"],
    "skat": ["MANDT", "SPRAS", "KTOPL", "SAKNR"],
    "csks": ["MANDT", "KOKRS", "KOSTL", "DATBI"],
    "cepc": ["MANDT", "KOKRS", "PRCTR", "DATBI"],
    "lfa1": ["MANDT", "LIFNR"],
    "kna1": ["MANDT", "KUNNR"],
}
FINANCE_MASTER_MAP = {
    "t001": {"MANDT": "RCLNT", "BUKRS": "RBUKRS"},
    "ska1": {"MANDT": "RCLNT", "SAKNR": "RACCT"},
    "skat": {"MANDT": "RCLNT", "SAKNR": "RACCT"},
    "csks": {"MANDT": "RCLNT", "KOKRS": "KOKRS", "KOSTL": "RCNTR"},
    "cepc": {"MANDT": "RCLNT", "KOKRS": "KOKRS", "PRCTR": "PRCTR"},
    "lfa1": {"MANDT": "RCLNT", "LIFNR": "LIFNR"},
    "kna1": {"MANDT": "RCLNT", "KUNNR": "KUNNR"},
}
PLAN_MASTER_MAP = {
    "t001": {"MANDT": "_CLIENT", "BUKRS": "COMPANY_CODE"},
    "ska1": {"MANDT": "_CLIENT", "SAKNR": "GL_ACCOUNT"},
    "skat": {"MANDT": "_CLIENT", "SAKNR": "GL_ACCOUNT"},
    "csks": {"MANDT": "_CLIENT", "KOKRS": "_KOKRS", "KOSTL": "COST_CENTER"},
    "cepc": {"MANDT": "_CLIENT", "KOKRS": "_KOKRS", "PRCTR": "PROFIT_CENTER"},
}


BUDGET_KEYS = [
    "COMPANY_CODE", "GL_ACCOUNT", "COST_CENTER", "PROFIT_CENTER",
    "FISCAL_YEAR", "FISCAL_PERIOD", "PLAN_VERSION",
]
PLAN_KEYS = ["RCLNT", "RLDNR", "RBUKRS", "RYEAR", "RACCT", "PRCTR", "VERSN"]
# Preserve legacy business columns; currency identity extends the deterministic key.
LEGACY_BUDGET_GRAIN = [
    "PLAN_SOURCE", "SCENARIO", "COMPANY_CODE", "GL_ACCOUNT", "COST_CENTER",
    "PROFIT_CENTER", "FISCAL_YEAR", "FISCAL_PERIOD", "PLAN_VERSION",
]
BUDGET_SOURCE_GRAIN = LEGACY_BUDGET_GRAIN + ["RCLNT", "RLDNR"]
grain = BUDGET_SOURCE_GRAIN + ["RHCUR"]
BUDGET_MIGRATION_RECOVERY = (
    "No Gold fact changes were applied. Restore authoritative Silver/T001 SCD history for the legacy grain, "
    "or back up and explicitly rebuild fct.budget after approving its client/ledger/currency mapping. "
    "Do not append new keys beside unresolved legacy rows."
)


def key_subset(rows, keys_df, keys: list[str], how: str = "left_semi"):
    left, right = rows.alias("_rows"), keys_df.select(*keys).distinct().alias("_keys")
    match = reduce(lambda a, b: a & b, [
        F.col(f"_rows.{key}").eqNullSafe(F.col(f"_keys.{key}")) for key in keys
    ])
    return left.join(right, match, how)


def read_change_keys(table_name: str, source, keys: list[str], change_type=None):
    journal = f"{AUD}.extraction_changes"
    if not spark.catalog.tableExists(journal):
        raise RuntimeError(f"Incremental Gold requires {journal}; do not infer deletes from batch IDs")
    changes = spark.table(journal).filter(
        (F.col("RUN_ID") == F.lit(run_id))
        & (F.lower(F.col("TABLE_NAME")) == F.lit(table_name))
    )
    invalid_type = ~F.col("CHANGE_TYPE").isin("INSERT", "UPDATE", "DELETE")
    if changes.filter(F.col("CHANGE_TYPE").isNull() | invalid_type).limit(1).count():
        raise ValueError(f"{table_name}: invalid journal CHANGE_TYPE")
    if change_type is not None:
        changes = changes.filter(F.col("CHANGE_TYPE") == F.lit(change_type))
    names = F.json_object_keys(F.col("KEY_JSON"))
    missing_key = reduce(lambda a, b: a | b, [
        ~F.array_contains(names, key) for key in keys
    ])
    if changes.filter(names.isNull() | missing_key).limit(1).count():
        raise ValueError(f"{table_name}: journal KEY_JSON must encode complete keys {keys}")
    key_schema = T.StructType([source.schema[key] for key in keys])
    return (changes.select(F.from_json("KEY_JSON", key_schema, {"mode": "FAILFAST"}).alias("_KEY"))
            .select("_KEY.*").dropDuplicates(keys))


def master_scope_keys(rows, scope_keys: list[str], planning: bool = False):
    """Repair affected natural keys, never replace historical lookups with IS_CURRENT."""
    journal = spark.table(f"{AUD}.extraction_changes").filter(F.col("RUN_ID") == F.lit(run_id))
    mappings = PLAN_MASTER_MAP if planning else FINANCE_MASTER_MAP
    affected = None
    for table_name, mapping in mappings.items():
        if not journal.filter(F.lower(F.col("TABLE_NAME")) == table_name).limit(1).count():
            continue
        source = spark.table(f"{SLV}.{table_name}")
        changed = read_change_keys(table_name, source, MASTER_SOURCE_KEYS[table_name])
        natural_keys = list(mapping)
        changed_keys = changed.select(*natural_keys).distinct()
        if table_name == "cepc":
            # Division is derived from CEPC. A changed division label/boundary
            # can also affect other profit centers sharing its old/new KHINR.
            profit_centers = spark.table(f"{DIM}.profit_center")
            changed_pc = key_subset(profit_centers, changed, MASTER_SOURCE_KEYS["cepc"])
            division_codes = changed_pc.select("KHINR").filter(F.col("KHINR").isNotNull()).distinct()
            related_pc = key_subset(profit_centers, division_codes, ["KHINR"]).select(*natural_keys).distinct()
            changed_keys = changed_keys.unionByName(related_pc).distinct()
        left, right = rows.alias("_mrows"), changed_keys.alias("_mkeys")
        conditions = []
        for master_key, row_key in mapping.items():
            condition = F.col(f"_mrows.{row_key}").eqNullSafe(F.col(f"_mkeys.{master_key}"))
            if planning and row_key == "_KOKRS":
                # Only an absent controlling area is unconstrained; client is always explicit.
                condition = F.col(f"_mrows.{row_key}").isNull() | condition
            conditions.append(condition)
        matching = left.join(right, reduce(lambda a, b: a & b, conditions), "left_semi").select(*scope_keys).distinct()
        affected = matching if affected is None else affected.unionByName(matching)
    return affected.distinct() if affected is not None else None


def business_key_hash(columns):
    return F.sha2(F.concat_ws("||", *[
        F.coalesce(column.cast("string"), F.lit("__NULL__")) for column in columns
    ]), 256)


def assert_fact_grain(source_count: int, output_count: int, duplicate_count: int, label: str) -> None:
    if duplicate_count or source_count != output_count:
        raise RuntimeError(f"{label} grain violation: source={source_count}, output={output_count}, duplicates={duplicate_count}; check dimension/client/ledger ambiguity")


def merge_fact(out, deleted_ids, target: str, key: str, additive_columns: tuple = ()):
    persisted = [column for column in out.columns if column != "_MATCH_SK"]
    if not spark.catalog.tableExists(target):
        out.select(*persisted).write.format("delta").mode("overwrite").saveAsTable(target)
        return
    target_columns = spark.table(target).columns
    missing = [column for column in persisted if column not in target_columns]
    if set(missing) - set(additive_columns):
        raise RuntimeError(f"{target}: unexpected schema additions {missing}; validate the deployed fact contract")
    match_column = "_MATCH_SK" if "_MATCH_SK" in out.columns else key
    matched_ids = out.select(F.col(match_column).alias(key)).unionByName(out.select(key)).distinct()
    removals = key_subset(deleted_ids.select(key).distinct(), matched_ids, [key], "left_anti")
    if match_column != key:
        removals = removals.withColumn(match_column, F.col(key))
    actions = out.withColumn("_DELETE", F.lit(False)).unionByName(
        removals.withColumn("_DELETE", F.lit(True)), allowMissingColumns=True
    )
    if actions.groupBy(match_column).count().filter(F.col("count") > 1).limit(1).count():
        raise RuntimeError(f"{target}: ambiguous re-key actions; back up and repair duplicate source identities before replay")
    values = {column: f"src.`{column}`" for column in persisted}
    compared = [column for column in persisted if column not in {"GOLD_BATCH_ID", "GOLD_LOAD_TIMESTAMP"}]
    changed = "true" if missing else "NOT (" + " AND ".join(f"tgt.`{column}` <=> src.`{column}`" for column in compared) + ")"
    builder = DeltaTable.forName(spark, target).alias("tgt").merge(
        actions.alias("src"), f"tgt.`{key}` = src.`{match_column}`"
    )
    if missing:
        if not hasattr(builder, "withSchemaEvolution"):
            raise RuntimeError("Additive Gold migration requires Delta 3.2+ merge schema evolution; no fact changes were applied")
        builder = builder.withSchemaEvolution()
    (builder.whenMatchedDelete(condition="src._DELETE")
     .whenMatchedUpdate(condition=f"NOT src._DELETE AND ({changed})", set=values)
     .whenNotMatchedInsert(condition="NOT src._DELETE", values=values).execute())


def dim_join(df, dim_name, alias, conditions, date_col="POSTING_DATE"):
    dimension = spark.table(f"{DIM}.{dim_name}").alias(alias)
    match = reduce(lambda left, right: left & right, conditions)
    date = F.col(date_col)
    match = match & date.between(F.col(f"{alias}.EFFECTIVE_FROM"), F.col(f"{alias}.EFFECTIVE_TO"))
    if dim_name in {"cost_center", "profit_center", "division"}:
        match = match & date.between(F.col(f"{alias}.DATAB"), F.col(f"{alias}.DATBI"))
    return df.join(dimension, match, "left")


def source_metadata(df):
    types = {"BATCH_RUN_ID": "string", "EXTRACT_DATE": "date", "LOAD_DATE": "date", "SOURCE_SYSTEM": "string", "SOURCE_FILE_NAME": "string"}
    return [(F.col(column).cast(data_type) if column in df.columns else F.lit(None).cast(data_type)).alias(column)
            for column, data_type in types.items()]


def budget_finance_context(config=None) -> tuple[str, str]:
    contract = globals().get("_SAP_EXTRACTION_CONFIG") if config is None else config
    if not isinstance(contract, dict) or not isinstance(contract.get("tables"), list):
        raise ValueError("BUDGET requires embedded _SAP_EXTRACTION_CONFIG.tables with explicit financeContext; no customer defaults are inferred")
    tables = [table for table in contract["tables"] if isinstance(table, dict) and str(table.get("name", "")).upper() == "BUDGET"]
    if len(tables) != 1 or not isinstance(tables[0].get("financeContext"), dict):
        raise ValueError("Exactly one BUDGET financeContext with explicit client and ledger is required")
    context = tables[0]["financeContext"]
    values = [context.get(key) for key in ["client", "ledger"]]
    if any(not isinstance(value, str) or not value.strip() for value in values):
        raise ValueError("BUDGET financeContext.client and .ledger must be non-empty strings; no inferred defaults are permitted")
    return values[0].strip(), values[1].strip()


def planning_date(year=None, period=None):
    year = F.col("FISCAL_YEAR") if year is None else year
    period = F.col("FISCAL_PERIOD") if period is None else period
    regular = F.make_date(year.cast("int"), F.greatest(F.lit(1), F.least(period.cast("int"), F.lit(12))), F.lit(1))
    return F.when(period.cast("int") <= 12, regular).otherwise(F.make_date(year.cast("int"), F.lit(12), F.lit(31)))


def assert_budget_unique(rows, keys: list[str], label: str):
    if rows.groupBy(*keys).count().filter(F.col("count") > 1).limit(1).count():
        raise RuntimeError(f"{label}: ambiguous budget identity {keys}. {BUDGET_MIGRATION_RECOVERY}")


def resolve_budget_currency(rows):
    # dim.company is the T001-derived SCD2 mapping; do not use today's WAERS for historical plans.
    left = rows.withColumn("_PLAN_DATE", planning_date()).alias("b")
    company = spark.table(f"{DIM}.company").alias("c")
    required = {"MANDT", "BUKRS", "WAERS", "EFFECTIVE_FROM", "EFFECTIVE_TO"}
    if required - set(company.columns):
        raise ValueError("BUDGET currency requires the complete T001-derived company SCD2 mapping")
    match = (F.col("b.RCLNT").eqNullSafe(F.col("c.MANDT"))
             & F.col("b.COMPANY_CODE").eqNullSafe(F.col("c.BUKRS"))
             & F.col("b._PLAN_DATE").between(F.col("c.EFFECTIVE_FROM"), F.col("c.EFFECTIVE_TO")))
    resolved = left.join(company, match, "left").select("b.*", F.trim(F.col("c.WAERS").cast("string")).alias("RHCUR")).drop("_PLAN_DATE")
    assert_budget_unique(resolved, BUDGET_SOURCE_GRAIN, "BUDGET_FILE source/as-of T001 mapping")
    if resolved.filter(F.col("RHCUR").isNull() | (F.col("RHCUR") == "")).limit(1).count():
        raise ValueError("BUDGET_FILE has no unique populated as-of T001.WAERS for its explicit client/company/period; repair company history, never infer a currency")
    return resolved


def budget_projection(df, values: bool = True):
    client, ledger = budget_finance_context()
    columns = [F.col(column) for column in BUDGET_KEYS if column != "PLAN_VERSION"] + [
        F.coalesce(F.col("PLAN_VERSION"), F.lit("BUDGET")).alias("PLAN_VERSION"),
        F.lit("BUDGET_FILE").alias("PLAN_SOURCE"), F.lit("BUDGET").alias("SCENARIO"),
        F.lit(client).cast("string").alias("RCLNT"), F.lit(ledger).cast("string").alias("RLDNR"),
    ]
    if values:
        columns += [F.col("BUDGET_AMOUNT_LOCAL").cast("decimal(23,2)").alias("PLAN_AMOUNT_LOCAL"),
                    F.col("FORECAST_AMOUNT_LOCAL").cast("decimal(23,2)").alias("FORECAST_AMOUNT_LOCAL"),
                    *source_metadata(df), F.lit(client).alias("_CLIENT"), F.lit(None).cast("string").alias("_KOKRS")]
    projected = df.select(*columns)
    return resolve_budget_currency(projected) if values else projected





def plan_projection(df, values: bool = True):
    periods = [F.struct(
        F.lit(period).alias("FISCAL_PERIOD"),
        (F.col(f"TSL{period:02d}").cast("decimal(23,2)") if values else F.lit(None).cast("decimal(23,2)")).alias("PLAN_AMOUNT_LOCAL"),
    ) for period in range(1, 17)]
    expanded = df.withColumn("_PERIOD", F.explode(F.array(*periods)))
    columns = [F.col("RBUKRS").alias("COMPANY_CODE"), F.col("RACCT").alias("GL_ACCOUNT"),
               F.lit(None).cast("string").alias("COST_CENTER"), F.col("PRCTR").alias("PROFIT_CENTER"),
               F.col("RYEAR").alias("FISCAL_YEAR"), F.col("_PERIOD.FISCAL_PERIOD").alias("FISCAL_PERIOD"),
               F.col("VERSN").alias("PLAN_VERSION"), F.lit("FAGLFLEXP").alias("PLAN_SOURCE"), F.lit("PLAN").alias("SCENARIO"),
               F.col("RCLNT").cast("string").alias("RCLNT"), F.col("RLDNR").cast("string").alias("RLDNR")]
    if values:
        if "RHCUR" not in df.columns:
            raise ValueError("Synthetic FAGLFLEXP projection requires its declared RHCUR; no currency is inferred")
        columns += [F.col("_PERIOD.PLAN_AMOUNT_LOCAL").alias("PLAN_AMOUNT_LOCAL"),
                    F.lit(None).cast("decimal(23,2)").alias("FORECAST_AMOUNT_LOCAL"), *source_metadata(df),
                    F.col("RCLNT").cast("string").alias("_CLIENT"),
                    (F.col("KOKRS") if "KOKRS" in df.columns else F.lit(None).cast("string")).alias("_KOKRS"),
                    F.trim(F.col("RHCUR").cast("string")).alias("RHCUR")]
    projected = expanded.select(*columns)
    if values:
        missing_identity = reduce(lambda a, b: a | b, [F.col(key).isNull() | (F.trim(F.col(key)) == "") for key in ["RCLNT", "RLDNR", "RHCUR"]])
        if projected.filter(missing_identity).limit(1).count():
            raise ValueError("Synthetic FAGLFLEXP rows require populated RCLNT/RLDNR/RHCUR")
    return projected


def budget_scope(all_budget, all_plan, target_exists: bool, force_full: bool = False):
    current = budget_projection(all_budget).unionByName(plan_projection(all_plan))
    assert_budget_unique(current, BUDGET_SOURCE_GRAIN, "Current Silver plan-source grain")
    if run_id and target_exists:
        budget_keys = read_change_keys("budget", all_budget, BUDGET_KEYS)
        plan_keys = read_change_keys("faglflexp", all_plan, PLAN_KEYS)
        affected = budget_projection(budget_keys, False).select(*BUDGET_SOURCE_GRAIN).unionByName(
            plan_projection(plan_keys, False).select(*BUDGET_SOURCE_GRAIN)
        ).distinct()
        master_keys = master_scope_keys(current, BUDGET_SOURCE_GRAIN, planning=True)
        if master_keys is not None:
            affected = affected.unionByName(master_keys).distinct()
        staged = current if force_full else key_subset(current, affected, BUDGET_SOURCE_GRAIN)
        delete_scope = budget_projection(read_change_keys("budget", all_budget, BUDGET_KEYS, "DELETE"), False).select(*BUDGET_SOURCE_GRAIN).unionByName(
            plan_projection(read_change_keys("faglflexp", all_plan, PLAN_KEYS, "DELETE"), False).select(*BUDGET_SOURCE_GRAIN)
        ).distinct()
        removed = key_subset(delete_scope, current, BUDGET_SOURCE_GRAIN, "left_anti")
    else:
        staged = current
        removed = current.select(*BUDGET_SOURCE_GRAIN).limit(0)
    return staged, removed


def budget_requires_migration(existing) -> bool:
    if set(grain) - set(existing.columns):
        return True
    missing_identity = reduce(lambda a, b: a | b, [F.col(key).isNull() | (F.trim(F.col(key).cast("string")) == "") for key in ["RCLNT", "RLDNR", "RHCUR"]])
    wrong_key = ~F.col("BUDGET_FACT_SK").eqNullSafe(business_key_hash([F.col(key) for key in grain]))
    return bool(existing.filter(missing_identity | wrong_key).limit(1).count())


def prepare_budget_upserts(out, existing, migrating: bool = False):
    """Return transient _MATCH_SK actions; one MERGE updates old SKs instead of appending duplicates."""
    assert_budget_unique(out, grain, "Curated budget grain")
    assert_budget_unique(out, BUDGET_SOURCE_GRAIN, "Curated budget source identity")
    assert_budget_unique(existing, ["BUDGET_FACT_SK"], "Existing budget surrogate keys")
    padded = existing
    for key in ["RCLNT", "RLDNR", "RHCUR"]:
        if key not in padded.columns:
            padded = padded.withColumn(key, F.lit(None).cast("string"))
    client, ledger = budget_finance_context()
    incompatible_context = (F.col("PLAN_SOURCE") == "BUDGET_FILE") & (
        (F.col("RCLNT").isNotNull() & (F.col("RCLNT") != client))
        | (F.col("RLDNR").isNotNull() & (F.col("RLDNR") != ledger)))
    if padded.filter(incompatible_context).limit(1).count():
        raise RuntimeError(f"Existing BUDGET_FILE identities conflict with explicit financeContext. {BUDGET_MIGRATION_RECOVERY}")
    if not migrating:
        assert_budget_unique(padded, BUDGET_SOURCE_GRAIN, "Existing budget source identity")
        lookup = padded.select(*BUDGET_SOURCE_GRAIN, F.col("BUDGET_FACT_SK").alias("_MATCH_SK")).alias("l")
        fresh = out.alias("n")
        match = reduce(lambda a, b: a & b, [F.col(f"n.{key}").eqNullSafe(F.col(f"l.{key}")) for key in BUDGET_SOURCE_GRAIN])
        return (fresh.join(lookup, match, "left").select("n.*", F.col("l._MATCH_SK").alias("_MATCH_SK"))
                .withColumn("_MATCH_SK", F.coalesce(F.col("_MATCH_SK"), F.col("BUDGET_FACT_SK"))))
    legacy_hash = business_key_hash([F.col(key) for key in LEGACY_BUDGET_GRAIN])
    full_hash = business_key_hash([F.col(key) for key in grain])
    missing = reduce(lambda a, b: a | b, [F.col(key).isNull() | (F.trim(F.col(key)) == "") for key in ["RCLNT", "RLDNR", "RHCUR"]])
    legacy_condition = missing | F.col("BUDGET_FACT_SK").eqNullSafe(legacy_hash)
    legacy = padded.filter(legacy_condition)
    modern = padded.filter(~legacy_condition)
    if modern.filter(~F.col("BUDGET_FACT_SK").eqNullSafe(full_hash)).limit(1).count():
        raise RuntimeError(f"Unrecognized existing budget key format. {BUDGET_MIGRATION_RECOVERY}")
    assert_budget_unique(legacy, LEGACY_BUDGET_GRAIN, "Legacy budget business grain")
    assert_budget_unique(modern, BUDGET_SOURCE_GRAIN, "Existing modern budget identity")
    left = legacy.select(*LEGACY_BUDGET_GRAIN, F.col("BUDGET_FACT_SK").alias("_LEGACY_SK"), *[
        F.col(key).alias(f"_OLD_{key}") for key in ["RCLNT", "RLDNR", "RHCUR"]
    ]).alias("l")
    right = out.alias("n")
    match = reduce(lambda a, b: a & b, [F.col(f"l.{key}").eqNullSafe(F.col(f"n.{key}")) for key in LEGACY_BUDGET_GRAIN])
    candidates = left.join(right, match, "left").select(
        F.col("l._LEGACY_SK").alias("_LEGACY_SK"), F.col("n.BUDGET_FACT_SK").alias("_NEW_SK"),
        *[F.col(f"l._OLD_{key}").alias(f"_OLD_{key}") for key in ["RCLNT", "RLDNR", "RHCUR"]],
        *[F.col(f"n.{key}").alias(key) for key in ["RCLNT", "RLDNR", "RHCUR"]])
    counts = candidates.groupBy("_LEGACY_SK").agg(F.count("_NEW_SK").alias("_CANDIDATES"))
    if counts.filter(F.col("_CANDIDATES") != 1).limit(1).count():
        raise RuntimeError(f"Legacy budget mapping has zero or multiple current client/ledger/currency candidates. {BUDGET_MIGRATION_RECOVERY}")
    conflict = reduce(lambda a, b: a | b, [F.col(f"_OLD_{key}").isNotNull() & ~F.col(f"_OLD_{key}").eqNullSafe(F.col(key)) for key in ["RCLNT", "RLDNR", "RHCUR"]])
    if candidates.filter(conflict).limit(1).count():
        raise RuntimeError(f"Legacy budget identity conflicts with the authoritative mapping. {BUDGET_MIGRATION_RECOVERY}")
    mapped = candidates.select(F.col("_NEW_SK").alias("BUDGET_FACT_SK"), F.col("_LEGACY_SK").alias("_MATCH_SK"))
    if key_subset(mapped, modern.select("BUDGET_FACT_SK"), ["BUDGET_FACT_SK"]).limit(1).count():
        raise RuntimeError(f"Legacy and modern budget rows overlap; refusing duplicate migration. {BUDGET_MIGRATION_RECOVERY}")
    return out.join(mapped, "BUDGET_FACT_SK", "left").withColumn("_MATCH_SK", F.coalesce(F.col("_MATCH_SK"), F.col("BUDGET_FACT_SK")))


def budget_deleted_ids(existing, removed):
    padded = existing
    for key in ["RCLNT", "RLDNR"]:
        if key not in padded.columns:
            padded = padded.withColumn(key, F.lit(None).cast("string"))
    # Currency is not in the Silver source key. Match the journal-confirmed
    # source identity to its existing Gold SK, never fabricate a deletion currency.
    return key_subset(padded, removed, BUDGET_SOURCE_GRAIN).select("BUDGET_FACT_SK").distinct()


target = f"{FCT}.budget"
all_budget = spark.table(f"{SLV}.budget")
all_plan = spark.table(f"{SLV}.faglflexp")
target_exists = spark.catalog.tableExists(target)
existing_budget = spark.table(target) if target_exists else None
currency_migration = target_exists and budget_requires_migration(existing_budget)
staged, removed = budget_scope(all_budget, all_plan, target_exists, force_full=currency_migration)

plan_date = planning_date()
resolved = staged.withColumn("_PLAN_DATE", plan_date).alias("s")
resolved = dim_join(resolved, "company", "co", [
    F.col("s.COMPANY_CODE").eqNullSafe(F.col("co.BUKRS")),
    F.col("s.RCLNT").eqNullSafe(F.col("co.MANDT")),
], "_PLAN_DATE")
resolved = dim_join(resolved, "gl_account", "gl", [
    F.col("s.GL_ACCOUNT").eqNullSafe(F.col("gl.SAKNR")), F.col("co.KTOPL").eqNullSafe(F.col("gl.KTOPL")),
    F.col("co.MANDT").eqNullSafe(F.col("gl.MANDT")),
], "_PLAN_DATE")
resolved = dim_join(resolved, "cost_center", "cc", [
    F.col("s.COST_CENTER").eqNullSafe(F.col("cc.KOSTL")), F.col("co.MANDT").eqNullSafe(F.col("cc.MANDT")),
    F.col("s._KOKRS").isNull() | F.col("s._KOKRS").eqNullSafe(F.col("cc.KOKRS")),
    F.col("s.COMPANY_CODE").eqNullSafe(F.col("cc.BUKRS")),
], "_PLAN_DATE")
resolved = dim_join(resolved, "profit_center", "pc", [
    F.col("s.PROFIT_CENTER").eqNullSafe(F.col("pc.PRCTR")), F.col("co.MANDT").eqNullSafe(F.col("pc.MANDT")),
    F.col("s._KOKRS").isNull() | F.col("s._KOKRS").eqNullSafe(F.col("pc.KOKRS")),
], "_PLAN_DATE")
resolved = dim_join(resolved, "division", "dv", [F.col("pc.KHINR").eqNullSafe(F.col("dv.DIVISION_CODE"))], "_PLAN_DATE")

out = resolved.select(
    business_key_hash([F.col(f"s.{column}") for column in grain]).alias("BUDGET_FACT_SK"),
    ((F.col("s.FISCAL_YEAR") * F.lit(100)) + F.col("s.FISCAL_PERIOD")).cast("int").alias("FISCAL_PERIOD_KEY"),
    F.col("co.DIMENSION_SK").alias("COMPANY_SK"), F.col("gl.DIMENSION_SK").alias("GL_ACCOUNT_SK"),
    F.col("cc.DIMENSION_SK").alias("COST_CENTER_SK"), F.col("pc.DIMENSION_SK").alias("PROFIT_CENTER_SK"), F.col("dv.DIMENSION_SK").alias("DIVISION_SK"),
    *[F.col(f"s.{column}").alias(column) for column in LEGACY_BUDGET_GRAIN],
    F.col("s.PLAN_AMOUNT_LOCAL"), F.col("s.FORECAST_AMOUNT_LOCAL"), F.col("s.BATCH_RUN_ID").alias("SOURCE_BATCH_RUN_ID"),
    F.col("s.EXTRACT_DATE"), F.col("s.LOAD_DATE").alias("SILVER_LOAD_DATE"), F.col("s.SOURCE_SYSTEM"), F.col("s.SOURCE_FILE_NAME"),
    F.lit(BATCH_ID).alias("GOLD_BATCH_ID"), F.lit(RUN_TIMESTAMP).alias("GOLD_LOAD_TIMESTAMP"),
    F.col("s.RCLNT"), F.col("s.RLDNR"), F.col("s.RHCUR"),
)
source_count = staged.count()
output_count = out.select("BUDGET_FACT_SK").distinct().count()
duplicate_count = out.groupBy("BUDGET_FACT_SK").count().filter(F.col("count") > 1).count()
assert_fact_grain(source_count, output_count, duplicate_count, "Budget")
if target_exists:
    prepared = prepare_budget_upserts(out, existing_budget, migrating=currency_migration)
    deleted_ids = budget_deleted_ids(existing_budget, removed)
else:
    prepared = out
    deleted_ids = out.select("BUDGET_FACT_SK").limit(0)
# Keep legacy audit amount/count names, but never aggregate across currency-less
# company/source/version groups. Each evidence row is one candidate Gold grain.
summary = (out.select(*grain, F.lit(1).cast("long").alias("ROW_COUNT"),
                      F.col("PLAN_AMOUNT_LOCAL").cast("decimal(33,2)").alias("TOTAL_PLAN_AMOUNT_LOCAL"))
           .withColumn("GOLD_BATCH_ID", F.lit(BATCH_ID)).withColumn("GOLD_LOAD_TIMESTAMP", F.lit(RUN_TIMESTAMP)))
summary.write.mode("append").option("mergeSchema", "true").format("delta").saveAsTable(f"{AUD}.budget_fact_reconciliation")
display(summary)

merge_fact(prepared, deleted_ids, target, "BUDGET_FACT_SK", additive_columns=("RCLNT", "RLDNR", "RHCUR"))
